# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LaibaSabir1/flyrank-ml-internship-laiba_sabir/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup

Rebuilds the same feature vector and Random Forest used in `w05_model.ipynb`
(`03_train_model.py`'s feature list: `MODEL_NUMERIC_FEATURES` / `MODEL_CATEGORICAL_FEATURES`
from `scripts/ml_utils.py`), so the before/after comparison below is apples-to-apples with my
Week-5 work.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/LaibaSabir1/flyrank-ml-internship-laiba_sabir"
REPO_DIR = "flyrank-ml-internship-laiba_sabir"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd, numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit

RAW_PATH = "data/raw/content_refresh_anonymized.csv"
assert os.path.exists(RAW_PATH), "starter CSV not found — are you at the repo root?"

df = pd.read_csv(RAW_PATH)
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])

MODEL_NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
MODEL_CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]

def build_feature_matrix(frame):
    num = frame[MODEL_NUMERIC_FEATURES].apply(pd.to_numeric, errors="coerce")
    num = num.replace([np.inf, -np.inf], np.nan).fillna(0)
    cat = frame[MODEL_CATEGORICAL_FEATURES].fillna("unknown").astype(str)
    enc = pd.get_dummies(cat, prefix=MODEL_CATEGORICAL_FEATURES, dtype=float)
    return pd.concat([num.reset_index(drop=True), enc.reset_index(drop=True)], axis=1)

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(y_true)[order[:k]].mean())

X = build_feature_matrix(df)
y = df["is_declining_label"].astype(int)
print(f"{len(df):,} rows | declining rate: {y.mean():.3f} | {X.shape[1]} feature columns")

30,000 rows | declining rate: 0.542 | 52 feature columns


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Two findings from the FlyRank starter results (`outputs/model_report.md` / the lane guide),
each with the methodology question I'd ask — the same question I now apply to my own work
in Section 2 and 3.

**Finding 1 — "The random forest reaches Precision@50 ≈ 0.74 vs. the baseline rule's 0.24, a
~3x lift."**
*My methodology question:* what split produced 0.74? The repo's own docs flag this directly —
`GUIDE.md`'s FAQ says the number is "library-version sensitive" and shifts with which rows land
at the 50th-place boundary. Precision@50 on 30,000 rows means the metric hinges on where ties
break among a fairly small top slice. Before trusting the 3x headline, I'd want to see it
survive a re-run with a different seed and a different client split — not just the one split
it was measured on.

**Finding 2 — "Top features are visibility- and freshness-related: `days_with_impressions`,
`log_impressions_90d`, `avg_position`, `content_age_days`."**
*My methodology question:* where does each of these come from, relative to the label? The
label is `trend_direction == "down"`, itself derived from `trend_pct`, which compares
`impressions_last_30d` vs `impressions_prev_30d`. `days_with_impressions` and
`log_impressions_90d` are 90-day totals that overlap the same 30-day window the label is built
from — they're not literally the label, but they share a lot of the same underlying signal
(both are "how much traffic did this page get recently"). That's not leakage in the strict
sense (the label isn't derived *from* them), but it does mean the model may be partly
rediscovering "pages with less recent traffic are more likely to be down," which is close to
tautological. Worth flagging as a directional-not-causal reading, not a discovery of a hidden
driver.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Week-5 used the pipeline's Random Forest (`class_weight="balanced_subsample"`, `max_depth=10`,
`min_samples_leaf=25`, `n_estimators=200`) on the same feature set built above. Here I compare
two splits on identical data, features, and model hyperparameters:

- **Naive split** — `train_test_split` stratified on the label only. Pages from the same
  client can land in both train and test.
- **Honest split** — `GroupShuffleSplit` grouped by `client_id`. No client's pages appear in
  both train and test (verified below), matching `scripts/03_train_model.py`'s
  `client_holdout` strategy.

In [2]:
RANDOM_STATE = 42
RF_PARAMS = dict(class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25,
                  n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE)

# --- Naive: stratified random split (ignores client_id) ---
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
rf_naive = RandomForestClassifier(**RF_PARAMS).fit(Xtr, ytr)
proba_naive = rf_naive.predict_proba(Xte)[:, 1]
p50_naive = precision_at_k(yte, proba_naive, 50)

# --- Honest: client-grouped split ---
groups = df["client_id"].astype(str)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
Xtr2, Xte2 = X.iloc[train_idx], X.iloc[test_idx]
ytr2, yte2 = y.iloc[train_idx], y.iloc[test_idx]
rf_honest = RandomForestClassifier(**RF_PARAMS).fit(Xtr2, ytr2)
proba_honest = rf_honest.predict_proba(Xte2)[:, 1]
p50_honest = precision_at_k(yte2, proba_honest, 50)

overlap = set(df.iloc[train_idx]["client_id"]) & set(df.iloc[test_idx]["client_id"])

print(f"NAIVE  (random, stratified)  Precision@50: {p50_naive:.3f}   test base rate: {yte.mean():.3f}")
print(f"HONEST (client-grouped)      Precision@50: {p50_honest:.3f}   test base rate: {yte2.mean():.3f}")
print()
print(f"Train clients: {df.iloc[train_idx]['client_id'].nunique()}  |  "
      f"Test clients: {df.iloc[test_idx]['client_id'].nunique()}  |  "
      f"Overlapping clients (must be 0): {len(overlap)}")

NAIVE  (random, stratified)  Precision@50: 0.900   test base rate: 0.542
HONEST (client-grouped)      Precision@50: 0.540   test base rate: 0.511

Train clients: 25  |  Test clients: 7  |  Overlapping clients (must be 0): 0


**Before/after, in words:** the naive split scores Precision@50 well above the honest
client-grouped split on this run. With only 32 clients total and ~7 landing in the held-out
group, a chunk of that naive-split number is the model re-recognizing *client-specific*
patterns it saw in training — not a page-level pattern that generalizes to a brand-new client.
The honest number is lower and noisier (it depends heavily on which 7 clients happen to land
in test), but it's the one that actually answers "would this work on a client we've never
seen?" — which is the real deployment question. This is the same "client-holdout" split
`03_train_model.py` and `outputs/model_report.md` use, so it's the number I'd report in the
capstone, not the naive one

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Repeating the Week-3 leakage hunt (`w03_feature_leakage_check.ipynb`) on the final Week-5
feature set. Two checks: (a) deliberately inject the known leaky column and watch the score
spike, then confirm it's excluded from the real feature set; (b) confirm no FlyRank product
flags snuck in (there are none in this dataset to begin with — the starter CSV ships
observable signals only).

In [3]:
# (a) Deliberately add trend_pct -- the column the label is derived from -- and watch what happens
X_leaky = X.copy()
X_leaky["trend_pct"] = pd.to_numeric(df["trend_pct"], errors="coerce").fillna(0)

Xtr3, Xte3 = X_leaky.iloc[train_idx], X_leaky.iloc[test_idx]
rf_leaky = RandomForestClassifier(**RF_PARAMS).fit(Xtr3, ytr2)
proba_leaky = rf_leaky.predict_proba(Xte3)[:, 1]
p50_leaky = precision_at_k(yte2, proba_leaky, 50)

top_importance = (
    pd.Series(rf_leaky.feature_importances_, index=X_leaky.columns)
    .sort_values(ascending=False)
    .head(5)
)

print(f"Client-grouped Precision@50 WITHOUT trend_pct: {p50_honest:.3f}")
print(f"Client-grouped Precision@50 WITH    trend_pct: {p50_leaky:.3f}   <- the leakage tell")
print()
print("Top 5 feature importances with trend_pct included:")
print(top_importance.to_string())

Client-grouped Precision@50 WITHOUT trend_pct: 0.540
Client-grouped Precision@50 WITH    trend_pct: 1.000   <- the leakage tell

Top 5 feature importances with trend_pct included:
trend_pct                0.781997
days_with_impressions    0.036697
log_impressions_90d      0.029874
avg_position             0.021695
content_age_days         0.019249


**Verdict:** adding `trend_pct` collapses the problem — precision jumps to a near-perfect
1.000 and `trend_pct` alone absorbs ~78% of feature importance, dwarfing every real signal.
That's the label in disguise: `trend_direction` (the label source) is computed directly from
`trend_pct`, so this isn't the model learning a pattern, it's the model reading the answer key.
Confirmed: **`trend_pct` and `trend_direction` are excluded from `MODEL_NUMERIC_FEATURES` /
`MODEL_CATEGORICAL_FEATURES`** in the actual Week-5 model — the check above just proves why
that exclusion matters, not that it was violated.

**Product-flag check:** the starter CSV (`data/raw/content_refresh_anonymized.csv`) never
contained `health_score`, `priority_score`, `action_type`, or `needs_ctr_fix` in the first
place — per `DATA_USE.md`, FlyRank's product decision flags are intentionally excluded from
this dataset. So there's nothing to accidentally leak in from that direction; the check here
is confirming absence, not removing something that crept in.

**Pseudonymous-ID check:** `content_id` and `client_id` are used only for grouping the split
above (`groups = df["client_id"]`) — neither appears in `MODEL_NUMERIC_FEATURES` /
`MODEL_CATEGORICAL_FEATURES`, so they were never candidate features to begin with.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

My boldest sentence from earlier notebooks (ML-02 / ML-07) was some version of: *"the model
beats the hand-written rule at picking pages to review."* That's close to defensible but
overreaches in two ways: it doesn't say *which* split the number came from, and "beats" reads
like a settled fact rather than a result from one run.

**Original (too bold):**
> "The random forest model beats the baseline rule, catching about 3x more of the true
> declining pages in the top 50."

**Rewritten (decision-support language):**
> "On a client-grouped holdout split, the Random Forest's Precision@50 was directionally
> higher than the transparent baseline rule's, though the honest-split gap is smaller and
> noisier than the naive-split gap — with only 32 clients total, the exact number is sensitive
> to which clients land in the test group. This is an observed, single-run comparison on the
> 30,000-row anonymized starter slice, not a guarantee that the model outperforms the rule on
> unseen clients or at warehouse scale. The model is a decision-support ranking aid for a
> reviewer's queue, not an automatic publishing decision, and it should be paired with the
> same client-holdout discipline every time it's re-evaluated."

The rewrite keeps the real result (the model does show a lift) but ties it to the specific
split, sample size, and single run it came from, and drops "beats" for "was directionally
higher than."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.